## Representation Plotting

- Input: $z_{enc}
- Tools: PCA (dimensionality, eigenvalue spectrum), ICC (trait vs state axes), UMAP/PHATE (nonlinear structure).

*"what geometric structure does the encoder learn?"*

In [ ]:
import numpy as np
from pathlib import Path

from src.analysis.eval_infra import (
    load_label, 
    load_escalation_labels, 
    compute_escalation_criterions)
from src.utils.io import (EXPERIMENTS_DIR, DATA_DIR, 
                          load_embeddings, load_sequences_dict, 
                          load_metadata, load_json, 
                          load_npz_dict)
from src.utils.seed import load_exp_seed, set_global_seed

In [ ]:
# -- Config Settings --
MODEL       = "test_01"
EMB_NAME    = "embeddings_40.npz"

In [ ]:
EXPERIMENTS     = Path("experiments")
MODEL_DIR       = EXPERIMENTS / MODEL
ANALYSIS_DIR    = MODEL_DIR / "analysis" / "representation"
FIGURES_DIR     = ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCES_PATH  = Path(DATA_DIR / "sequences.jsonl")

set_global_seed(load_exp_seed(EXPERIMENTS_DIR / MODEL))

# -- Load embeddings --
emb, EMB_PATH = load_embeddings(MODEL_DIR, EMB_NAME)
print(f"Embeddings: {EMB_PATH.name}")

z_encs = emb["z_encs"]             # (N, C_padded, D)
z_pred = emb["z_pred"]             # (N, D)
z_target = emb["z_target"]         # (N, D)
ctx_pad_masks = emb["ctx_pad_masks"]  # (N, C_padded)
subject_ids = emb["subject_ids"]   # (N,)
mask_pos = emb["mask_pos"]         # (N,)
pred_error = z_pred - z_target     # (N, D)

# Flatten valid encounters from z_encs
valid = ~ctx_pad_masks.astype(bool)
z_enc_flat = z_encs[valid]         # (N_valid, D)
enc_subject_ids = np.broadcast_to(
    subject_ids[:, None], ctx_pad_masks.shape)[valid]
enc_positions = np.broadcast_to(
    np.arange(ctx_pad_masks.shape[1])[None, :], ctx_pad_masks.shape)[valid]

print(f"  z_encs:      {z_encs.shape}")
print(f"  z_enc_flat:  {z_enc_flat.shape}")
print(f"  z_pred:      {z_pred.shape}")
print(f"  z_target:    {z_target.shape}")
print(f"  pred_error:  {pred_error.shape}")
print(f"  subjects:    {len(np.unique(subject_ids))}")

# -- Load labels from sequences.jsonl
patients = load_sequences_dict(SEQUENCES_PATH)
unique_sids = np.unique(subject_ids)

label_escalation_patient = load_label(patients, unique_sids, "label_escalation")
label_30d_patient = load_label(patients, unique_sids, "label_30d")
label_esc_per_sample = load_escalation_labels(patients, subject_ids, mask_pos)
criteria_labels = compute_escalation_criterions(patients, subject_ids, mask_pos)
 
print(f"  Escalation rate (patient): {label_escalation_patient.mean():.3f}")
print(f"  Escalation rate (encounter): {label_esc_per_sample.mean():.3f}")
print(f"  30d readmit rate: {label_30d_patient.mean():.3f}")

In [ ]:
try:
    meta, meta_names, meta_pids = load_metadata(DATA_DIR)
    print(f"  Metadata: {meta.shape[0]} patients x {meta.shape[1]} features")
except FileNotFoundError:
    print("  Metadata not found")

In [ ]:

results = load_json(ANALYSIS_DIR / "representation.json")
if results is None:
    raise FileNotFoundError(f"No representation results found")
pca_enc = load_npz_dict(ANALYSIS_DIR / "pca_encounter.npz")
pca_pat = load_npz_dict(ANALYSIS_DIR / "pca_patient.npz")
umap_emb = load_npz_dict(ANALYSIS_DIR / "umap.npy")
phate_emb = load_npz_dict(ANALYSIS_DIR / "phate.npy")
cluster_labels = load_json(ANALYSIS_DIR / "clusters.json")
sae_data = load_json(ANALYSIS_DIR / "sae_features.json")

### Plotting

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

*first figure reviewers see - it establishes that the latent space has structure.*

In [ ]:
from src.analysis.plotting import _s1_eigenvalue_spectrum
_s1_eigenvalue_spectrum(results, pca_enc, pca_pat,
                        show=True, save=False, fig_dir=FIGURES_DIR)

- rows = top-k PC axes, single column of ICC values
- Small figure, meant to be a panel in a composite.

In [ ]:
from src.analysis.plotting import _s1_icc_heatmap
_s1_icc_heatmap(results, show=True, save=False, fig_dir=FIGURES_DIR)

### UMAP on $z_{enc}$

*Uniform Manifold Approximation and Projection for Dimension Reduction*

UMAP constructs a fuzzy weighted graph approximating the topological structure of the high-dimensional point cloud, then finds a low-dimensional embedding that preserves the topology by balancing attraction between nearby points against repulsion between distant points. 

UMAP excels at revealing *discrete cluster structure* and *local density variation*. The space between and shapes of clusters is less meaningful as continuous gradients in the data get increasingly discretized, however UMAP is the primary substrate for HDBSCAN cluster enrichment analysis -  the sharp cluster boundaries make it the natural choice for identifying discrete patient subpopulations and testing whether each cluster has a coherent clinical signature (The clusters that emerge from UMAP space define the neighborhoods against which SAE features are cross-referenced in. a SAE feature that concentrates in a single UMAP cluster is capturing something the nonlinear topology already identifies, while a SAE feature that cuts across multiple clusters is finding finer-grained structure that UMAP's discrete topology doesn't resolve.)

Four views of the $z_{enc}$ UMAP embedding:
- (a) colored by HDBSCAN cluster assignment
- (b) colored by escalation label (binary)
- (c) colored by escalation criterion (categorical - which criterion fired, for escalation=1 samples; gray for escalation=0)
- (d) colored by encounter position in sequence (ordinal - early encounters vs late encounters)

In [ ]:
from src.analysis.plotting import _s1_embedding_panel
if umap_emb is not None:
    _s1_embedding_panel(
        umap_emb, cluster_labels, [0,1], criteria_labels, mask_pos,
        "UMAP", show=True, save=False, fig_dir=FIGURES_DIR)


### PHATE on $z_{enc}$

*Potential of Heat-diffusion for Affinity-based Trajectory Embedding*

PHATE reveals trajectory structure that UMAP doesn't.

PHATE builds a diffusion operator (transition matrix) over the point cloud. It encodes the probability of reaching any point from any other point via a random walk on the data manifold, then applies a potential distance transform to convert the probabilities into distances that preserve both neighborhood structure and global trajectory geometry. The result is a 2D embedding where euclidean distance approximates diffusion distance on the underlying manifold. 

This makes PHATE particularly well-suited for data with trajectory-like structure - gradual clinical transitions between patient states are rendered as smooth, continuous paths in the embedding rather than being shattered into discrete clumps. Where UMAP acts as in identification, PHATE acts to identify state change among the global embedding geometry. Patients that are far apart in PHATE space are genuinely far apart in terms of the dynamics encoded in the latent representation.

In [ ]:
if phate_emb is not None:
    _s1_embedding_panel(
        phate_emb, cluster_labels, [0,1], criteria_labels, mask_pos,
        "PHATE", show=True, save=False, fig_dir=FIGURES_DIR)

side-by-side bars for each top-k PC: encounter-level $R^2$ vs patient-level $R^2$. Annotate the mean across PCs. Gap between them tells the encounter-vs-patient-identity story.

In [ ]:
from src.analysis.plotting import _s1_lasso_r2_bar
_s1_lasso_r2_bar(results, show=True, save=False, fig_dir=FIGURES_DIR)

Grid of top-N features (by activation fraction). Each card shows:
- feature index,
- activation fraction
- top 3 enriched ICD codes with odds ratios,
- top 3 enriched meds with odds ratios, 
- classification label (clinical match / mixed / no match).

Render as a matplotlib figure, not text.

In [ ]:
from src.analysis.plotting import _s1_sae_feature_cards
if sae_data and sae_data.get("feature_cards"):
    _s1_sae_feature_cards(sae_data, show=True, save=False, fig_dir=FIGURES_DIR)

(if both SAE and clusters exist)
- rows = SAE features (active ones)
- columns = HDBSCAN clusters
- Cell color = mean activation of that feature in that cluster.
- Annotate which features are concentrated in single clusters vs spread across many.

In [ ]:
from src.analysis.plotting import _s1_sae_cluster_crossref
if sae_data and sae_data.get("feature_cards") and cluster_labels is not None:
    _s1_sae_cluster_crossref(sae_data, show=True, save=False, fig_dir=FIGURES_DIR)